# Olentangy salaries — silver layer

Queries the DuckDB built by `make build`. Three tables:

| table | grain | notes |
|---|---|---|
| `salaries` | one row per employee | includes names — local only, never exported |
| `salary_roles` | one row per employee-role | the unnested `Position` field |
| `salaries_gold` | one row per employee | publish shape, names dropped |

The connection is **read-only** so this notebook can't take the write lock away from a
Dagster run. Re-run `make build` in a terminal and just re-execute cells here.

In [1]:
from pathlib import Path

import duckdb

# walk up from wherever this runs, so it works from notebooks/ or the project root
REL = "data/silver/salaries.duckdb"
here = Path.cwd().resolve()
DB = next((p / REL for p in (here, *here.parents) if (p / REL).exists()), None)
assert DB, f"no {REL} found at or above {here} — run `make build` first"

con = duckdb.connect(DB, read_only=True)
con.sql("SHOW TABLES")

┌───────────────┐
│     name      │
│    varchar    │
├───────────────┤
│ salaries      │
│ salaries_gold │
│ salary_roles  │
└───────────────┘

In [2]:
con.sql("""
    SELECT count(*) AS employees,
           round(sum(gross_pay)) AS total_payroll,
           round(median(gross_pay)) AS median_pay,
           max(gross_pay) AS max_pay
    FROM salaries
""")

┌───────────┬───────────────┬───────────────┬───────────────┐
│ employees │ total_payroll │  median_pay   │    max_pay    │
│   int64   │ decimal(38,0) │ decimal(12,0) │ decimal(12,2) │
├───────────┼───────────────┼───────────────┼───────────────┤
│      4238 │     236230752 │         49149 │     291734.99 │
└───────────┴───────────────┴───────────────┴───────────────┘

## Role grain

88 raw roles, parsed into `role_base` + `contract_days` + `role_suffix`. Grouping by
`role_base` collapses contract variants (`SECRETARY 183/226/260` → `SECRETARY`) and the
filter drops `-0` lines, which are low-pay secondary records.

Some labels are plain titles (`TEACHER`), others are undocumented district payroll codes
(`CMF`, `CLAS-NON`, `LT-PLCMT`). Mapping those to job families is the open modeling question —
deliberately not answered here.

In [3]:
con.sql("""
    SELECT r.role_base,
           count(*) AS employees,
           round(median(s.gross_pay)) AS median_pay
    FROM salary_roles r
    JOIN salaries s USING (employee_id)
    WHERE r.role_suffix IS NULL
    GROUP BY 1
    ORDER BY employees DESC
    LIMIT 20
""")

┌─────────────────────────┬───────────┬───────────────┐
│        role_base        │ employees │  median_pay   │
│         varchar         │   int64   │ decimal(12,0) │
├─────────────────────────┼───────────┼───────────────┤
│ TEACHER                 │      1724 │         98099 │
│ SUB-CLAS                │       468 │          4276 │
│ SUP-ATHL                │       360 │          4144 │
│ LT-PLCMT                │       355 │         11528 │
│ INT-AIDE                │       311 │         33624 │
│ INTERVENTION SPECIALIST │       282 │         87491 │
│ INTERVENTION AIDE       │       274 │         33362 │
│ CMF                     │       195 │         56171 │
│ ADMIN                   │       191 │         88929 │
│ DRIVER                  │       189 │         36408 │
│ CLAS-NON                │       183 │         38604 │
│ FOOD-SVC                │       173 │         20383 │
│ TRANS DRIVER            │       163 │         38557 │
│ SECRETARY               │       117 │         

## Employees holding several roles

1,662 of 4,238 (39%). Note the caveat: `gross_pay` is the employee's *total*, not pay
attributable to any one of their roles, so don't sum it per-role without double-counting.

In [4]:
con.sql("""
    SELECT s.positions,
           count(*) AS employees,
           round(median(s.gross_pay)) AS median_pay
    FROM salaries s
    WHERE s.positions LIKE '%;%'
    GROUP BY 1
    ORDER BY employees DESC
    LIMIT 15
""")

┌───────────────────────────────────────┬───────────┬───────────────┐
│               positions               │ employees │  median_pay   │
│                varchar                │   int64   │ decimal(12,0) │
├───────────────────────────────────────┼───────────┼───────────────┤
│ INTERVENTION SPECIALIST; TEACHER      │       229 │         85477 │
│ INT-AIDE; INTERVENTION AIDE           │       209 │         35256 │
│ DRIVER; TRANS DRIVER                  │       150 │         39128 │
│ CMF; CUSTODIAN                        │       113 │         59862 │
│ FOOD SVC WORKER; FOOD-SVC             │        98 │         20074 │
│ LT-PLCMT; SUB-CLAS                    │        82 │         10781 │
│ EL SPECIALIST; TEACHER                │        45 │        101467 │
│ TEACHER; TEACHER PT/JS                │        37 │         46882 │
│ CLAS-NON; SECRETARY 183               │        35 │         40979 │
│ CLAS-NON; MONITOR/AIDE                │        29 │         28818 │
│ LITERACY SPECIALIS

## Into pandas

`.df()` on any query hands you a DataFrame for plotting or modeling.

In [5]:
df = con.sql("SELECT * FROM salaries_gold").df()
df["gross_pay"] = df["gross_pay"].astype(float)
df.describe()

,employee_id,contract_days,gross_pay
count,4238.000000,176.0,4238.000000
mean,2119.500000,215.573864,55741.093039
std,1223.549549,31.261207,43634.398084
min,1.000000,183.0,25.630000
25%,1060.250000,185.0,13381.135000
50%,2119.500000,200.0,49148.585000
75%,3178.750000,260.0,94987.640000
max,4238.000000,260.0,291734.990000
